# NYC Yellow Taxi 2019: a PySpark Data Processing Pipeline

**Data:** every NYC yellow taxi trip from 2019 (12 gzipped CSV files in `/databricks-datasets/nyctaxi/`), joined with the NYC TLC taxi-zone and payment-type lookup tables.

**Pipeline:** raw CSV → filter and clean → enrich with `withColumn` → Delta table partitioned by month → joins, aggregations and SQL → Parquet output.

Run on Databricks Free Edition (Serverless). Run the cells top to bottom.

## 0. Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               DoubleType, StringType, TimestampType)
from datetime import datetime
import time

CATALOG, SCHEMA = "workspace", "default"
TRIPS_TABLE  = f"{CATALOG}.{SCHEMA}.taxi_trips_2019"
VOLUME_NAME  = "taxi_output"
OUTPUT_DIR   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

DATA_DIR     = "/databricks-datasets/nyctaxi"
TRIPS_PATH   = f"{DATA_DIR}/tripdata/yellow/yellow_tripdata_2019-*.csv.gz"
ZONES_PATH   = f"{DATA_DIR}/taxizone/taxi_zone_lookup.csv"
PAYMENT_PATH = f"{DATA_DIR}/taxizone/taxi_payment_type.csv"

def timed(label, fn):
    """Run fn(), print how long it took, return its result."""
    start = time.time()
    result = fn()
    print(f"{label}: {time.time() - start:.2f} s")
    return result

## 1. Dataset size and schema check

The assignment asks for 1 GB+. The files are gzip-compressed, so the uncompressed CSV is several times larger than the number below.

In [ ]:
files = [f for f in dbutils.fs.ls(f"{DATA_DIR}/tripdata/yellow/")
         if f.name.startswith("yellow_tripdata_2019")]
for f in files:
    print(f"  {f.name:<34} {f.size / 1e6:>8,.0f} MB")
print(f"\n{len(files)} files, {sum(f.size for f in files) / 1e9:.2f} GB compressed")

In [ ]:
# All 12 monthly files should share the same header (we apply one schema to all of them).
for m in ["01", "06", "12"]:
    cols = spark.read.option("header", True).csv(
        f"{DATA_DIR}/tripdata/yellow/yellow_tripdata_2019-{m}.csv.gz").columns
    print(m, len(cols), cols)

In [ ]:
print(dbutils.fs.head(ZONES_PATH, 300))
print(dbutils.fs.head(PAYMENT_PATH, 300))

## 2. Load the data

We give an **explicit schema** instead of `inferSchema=True`. Inferring would make Spark read every row of 12 large files just to guess types; with a schema, defining the DataFrame costs nothing.

In [ ]:
trip_schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            IntegerType()),
    StructField("store_and_fwd_flag",    StringType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("extra",                 DoubleType()),
    StructField("mta_tax",               DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("tolls_amount",          DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount",          DoubleType()),
    StructField("congestion_surcharge",  DoubleType()),
])

raw = (spark.read
       .option("header", True)
       .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
       .schema(trip_schema)
       .csv(TRIPS_PATH))

zones = (spark.read.option("header", True)
         .schema("LocationID INT, Borough STRING, Zone STRING, service_zone STRING")
         .csv(ZONES_PATH))

payment_types = (spark.read.option("header", True)
                 .schema("payment_type INT, payment_desc STRING")
                 .csv(PAYMENT_PATH))

raw.printSchema()

In [ ]:
display(raw.limit(5))
display(zones.limit(5))
display(payment_types)

## 3. Actions vs transformations (lazy vs eager)

**Transformations** (`filter`, `withColumn`, `groupBy`, `join`, `select`) only add steps to a logical plan; they return instantly and read no data.
**Actions** (`count`, `show`, `collect`, `write`) make Spark optimize the plan and actually run jobs on the cluster.

The first cell below chains four transformations over ~7 GB of CSV and returns in milliseconds. The second cell runs an action on the same DataFrame and takes much longer.
Open **See performance** under each cell: the first cell shows no Spark jobs; the second shows the jobs that ran.

In [ ]:
t0 = time.time()
lazy_df = (raw
           .filter(F.col("fare_amount") > 0)                                       # transformation
           .withColumn("tip_pct", F.expr("try_divide(tip_amount, fare_amount) * 100"))  # transformation
           .groupBy("payment_type")                                                # transformation
           .agg(F.count("*").alias("trips"),
                F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct")))                 # transformation
print(f"4 transformations defined in {time.time() - t0:.4f} s. No data has been read yet.")
print(lazy_df)

In [ ]:
# ACTION: this is where Spark actually reads all 12 files.
timed("Action: show() on the grouped result", lambda: lazy_df.orderBy("payment_type").show())

In [ ]:
# ACTION: count every raw row (also used later to measure how many rows cleaning removed).
raw_rows = timed("Action: count() raw CSV rows", raw.count)
print(f"{raw_rows:,} raw trip records")

## 4. Clean and enrich (filters + `withColumn`)

**Filters early:** the two filters on raw columns come first, so Spark can apply them while it scans the CSV and every later step processes fewer rows.
The third filter needs derived columns, so it comes after them.
We finish with `select` so only the columns we need are kept (**column pruning**).

`try_divide` returns `null` instead of failing when it divides by zero (serverless runs in ANSI mode, where `x / 0` raises an error).

In [ ]:
clean = (raw
    # FILTER 1: keep only trips that actually start in 2019 (the raw files contain some bad timestamps)
    .filter((F.col("tpep_pickup_datetime") >= F.lit(datetime(2019, 1, 1))) &
            (F.col("tpep_pickup_datetime") <  F.lit(datetime(2020, 1, 1))))
    # FILTER 2: physically plausible trips
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) &
            (F.col("fare_amount") >= 2.5) & (F.col("fare_amount") < 500) &   # $2.50 = NYC minimum fare
            F.col("passenger_count").between(1, 6) &
            F.col("PULocationID").isNotNull() & F.col("DOLocationID").isNotNull())
    # COLUMN TRANSFORMATIONS
    .withColumn("duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
    .withColumn("is_weekend",   F.dayofweek("tpep_pickup_datetime").isin(1, 7))   # 1 = Sunday, 7 = Saturday
    .withColumn("avg_mph",      F.round(F.expr("try_divide(trip_distance, duration_min / 60)"), 2))
    .withColumn("fare_per_mile", F.round(F.expr("try_divide(fare_amount, trip_distance)"), 2))
    # Tips are only recorded for card payments (payment_type = 1), so tip % is null otherwise.
    # Tips above 100% of the fare are treated as data-entry errors (null).
    .withColumn("tip_pct", F.when(F.col("payment_type") == 1,
                                  F.round(F.expr("try_divide(tip_amount, fare_amount)") * 100, 2)))
    .withColumn("tip_pct", F.when(F.col("tip_pct").between(0, 100), F.col("tip_pct")))
    # FILTER 3: drop impossible durations and speeds (needs the derived columns)
    .filter(F.col("duration_min").between(1, 180) & (F.col("avg_mph") < 80))
    # COLUMN PRUNING: keep only what downstream steps use
    .select(F.col("tpep_pickup_datetime").alias("pickup_ts"),
            F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
            "pickup_month", "pickup_hour", "is_weekend",
            "PULocationID", "DOLocationID", "passenger_count", "payment_type",
            "trip_distance", "duration_min", "avg_mph",
            "fare_amount", "tip_amount", "tip_pct", "fare_per_mile", "total_amount"))

# Still lazy: nothing has run. Look for the filters inside the scan (PushedFilters).
clean.explain()

### Write the cleaned data as a Delta table, partitioned by month

Partitioning by `pickup_month` stores each month in its own folder. Any query that filters on month then reads only those folders (**partition pruning**).
Month is a good partition key: there are 12 values of roughly equal size, it shows up in many filters, and it does not create thousands of tiny files the way `pickup_ts` would.

This step converts slow, unsplittable `.csv.gz` into columnar Delta, so every later step reads Delta instead of re-parsing CSV.

In [ ]:
timed("Write clean trips to Delta (partitioned by pickup_month)",
      lambda: (clean.write
               .mode("overwrite")
               .option("overwriteSchema", "true")
               .partitionBy("pickup_month")
               .saveAsTable(TRIPS_TABLE)))

trips = spark.table(TRIPS_TABLE)
clean_rows = trips.count()
removed = raw_rows - clean_rows
print(f"{clean_rows:,} clean trips kept; {removed:,} removed ({removed / raw_rows:.1%}) by the filters")

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {TRIPS_TABLE}")
        .select("format", "partitionColumns", "numFiles", "sizeInBytes"))

## 5. Optimization: CSV vs partitioned Delta

The same question ("how many trips started at JFK in June?") asked two ways:
* **CSV pipeline:** `pickup_month` is computed from the timestamp, so Spark must decompress and parse **all 12 files**.
* **Delta table:** Spark skips 11 of the 12 month folders (`PartitionFilters`) and reads only the 2 columns it needs.

In [ ]:
JFK = 132
q_csv   = clean.filter((F.col("pickup_month") == 6) & (F.col("PULocationID") == JFK))
q_delta = trips.filter((F.col("pickup_month") == 6) & (F.col("PULocationID") == JFK))

n_csv   = timed("CSV   (re-parse all 12 gz files)", q_csv.count)
n_delta = timed("Delta (partition-pruned to June)  ", q_delta.count)
print(f"Both return the same answer: {n_csv:,} vs {n_delta:,}")

In [ ]:
# Look for PartitionFilters: [... (pickup_month = 6)] and the short ReadSchema
q_delta.explain()

## 6. Joins and groupBy aggregations

The zone and payment lookup tables are tiny (265 and 6 rows), so we **broadcast** them: each executor gets a full copy and the large trips table is never shuffled for the join.

In [ ]:
pu_zones = zones.select(F.col("LocationID").alias("PULocationID"),
                        F.col("Borough").alias("pu_borough"),
                        F.col("Zone").alias("pu_zone"))
do_zones = zones.select(F.col("LocationID").alias("DOLocationID"),
                        F.col("Borough").alias("do_borough"),
                        F.col("Zone").alias("do_zone"))

trips_enriched = (trips
    .join(F.broadcast(pu_zones),      on="PULocationID", how="left")
    .join(F.broadcast(do_zones),      on="DOLocationID", how="left")
    .join(F.broadcast(payment_types), on="payment_type", how="left"))

# GROUP BY 1: trip profile by pickup borough
borough_stats = (trips_enriched
    .groupBy("pu_borough")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("trip_distance"), 2).alias("avg_miles"),
         F.round(F.avg("duration_min"), 1).alias("avg_minutes"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct_card"),
         F.round(F.avg("avg_mph"), 1).alias("avg_mph"))
    .orderBy(F.desc("trips")))
display(borough_stats)

In [ ]:
# Look for BroadcastHashJoin (no shuffle of the trips table) and the partial/final aggregation around one Exchange
borough_stats.explain()

In [ ]:
# GROUP BY 2 (complex aggregation): how traffic speed and price per mile change by hour, weekday vs weekend
hourly_patterns = (trips
    .groupBy("is_weekend", "pickup_hour")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("avg_mph"), 1).alias("avg_mph"),
         F.round(F.percentile_approx("duration_min", 0.5), 1).alias("median_minutes"),
         F.round(F.avg("fare_per_mile"), 2).alias("avg_fare_per_mile"),
         F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct_card"))
    .orderBy("is_weekend", "pickup_hour"))
display(hourly_patterns)

In [ ]:
# GROUP BY 3: how people pay, by borough (uses the payment-type join)
payment_mix = (trips_enriched
    .groupBy("pu_borough")
    .pivot("payment_desc")
    .count()
    .orderBy("pu_borough"))
display(payment_mix)

## 7. SQL queries

The cleaned table is already in Unity Catalog; we register the lookup tables as temporary views so SQL can join to them.

In [ ]:
zones.createOrReplaceTempView("zones")
payment_types.createOrReplaceTempView("payment_types")

In [ ]:
# SQL 1: the 10 busiest pickup zones
top_zones = spark.sql(f"""
    SELECT z.Borough,
           z.Zone,
           COUNT(*)                     AS trips,
           ROUND(AVG(t.fare_amount), 2) AS avg_fare,
           ROUND(AVG(t.tip_pct), 1)     AS avg_tip_pct_card
    FROM {TRIPS_TABLE} t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough, z.Zone
    ORDER BY trips DESC
    LIMIT 10
""")
display(top_zones)

In [ ]:
# SQL 2: airport pickups by month (JFK = 132, LaGuardia = 138, Newark = 1)
airport_monthly = spark.sql(f"""
    SELECT CASE PULocationID WHEN 132 THEN 'JFK'
                             WHEN 138 THEN 'LaGuardia'
                             WHEN 1   THEN 'Newark' END AS airport,
           pickup_month,
           COUNT(*)                      AS trips,
           ROUND(AVG(total_amount), 2)   AS avg_total_paid,
           ROUND(AVG(duration_min), 1)   AS avg_minutes,
           ROUND(AVG(trip_distance), 1)  AS avg_miles
    FROM {TRIPS_TABLE}
    WHERE PULocationID IN (1, 132, 138)
    GROUP BY 1, 2
    ORDER BY airport, pickup_month
""")
display(airport_monthly)

In [ ]:
# SQL 3 (window function): the single busiest route starting in each borough
top_route_per_borough = spark.sql(f"""
    WITH routes AS (
        SELECT pz.Borough AS pu_borough,
               pz.Zone    AS pu_zone,
               dz.Zone    AS do_zone,
               COUNT(*)   AS trips
        FROM {TRIPS_TABLE} t
        JOIN zones pz ON t.PULocationID = pz.LocationID
        JOIN zones dz ON t.DOLocationID = dz.LocationID
        WHERE t.PULocationID <> t.DOLocationID
        GROUP BY pz.Borough, pz.Zone, dz.Zone
    )
    SELECT pu_borough, pu_zone, do_zone, trips
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY pu_borough ORDER BY trips DESC) AS rn
          FROM routes)
    WHERE rn = 1
    ORDER BY trips DESC
""")
display(top_route_per_borough)

In [ ]:
top_route_per_borough.explain()

## 8. More optimization experiments

### 8a. Filter placement: Catalyst pushes filters down for us
Below, the same query is written two ways: filtering **after** the join, and filtering **before** it.
Compare the two physical plans: they should be the same, because Catalyst's *predicate pushdown* rule moves the filter below the join and into the scan.

In [ ]:
filter_late = (trips
    .join(F.broadcast(pu_zones), "PULocationID")
    .filter((F.col("pickup_month") == 12) & (F.col("pu_borough") == "Manhattan"))
    .groupBy("pu_zone").count())

filter_early = (trips
    .filter(F.col("pickup_month") == 12)
    .join(F.broadcast(pu_zones.filter(F.col("pu_borough") == "Manhattan")), "PULocationID")
    .groupBy("pu_zone").count())

print("=== Filter written AFTER the join ===")
filter_late.explain()
print("=== Filter written BEFORE the join ===")
filter_early.explain()

### 8b. Broadcast join vs shuffle (sort-merge) join
We force a sort-merge join with a hint to see what the broadcast saves: a sort-merge join has to **shuffle** both sides by the join key.
(With Adaptive Query Execution, Spark can sometimes switch to a broadcast join at runtime anyway; check the plans.)

In [ ]:
smj = trips.join(pu_zones.hint("shuffle_merge"), "PULocationID").groupBy("pu_borough").count()
bhj = trips.join(F.broadcast(pu_zones),          "PULocationID").groupBy("pu_borough").count()

timed("Sort-merge join (shuffles 80M+ rows)", smj.collect)
timed("Broadcast hash join (no shuffle of trips)", bhj.collect)

In [ ]:
print("=== Sort-merge join plan ===")
smj.explain()
print("=== Broadcast join plan ===")
bhj.explain()

### 8c. Column pruning
Delta is columnar, so selecting 2 columns reads only those 2 columns from disk. Look at `ReadSchema`.

In [ ]:
trips.select("pickup_month", "fare_amount").groupBy("pickup_month").avg("fare_amount").explain()

## 9. Caching (bonus)

When the same intermediate result feeds several actions, Spark recomputes it from the source every time, unless we cache it.
We run 3 actions on a summer subset without caching, then with caching.

> Note: Databricks **serverless** compute may not support `.cache()`/`.persist()`. If it raises an error, the cell catches it, and instead **materializes** the subset as a Delta table, which is the serverless way to reuse an expensive intermediate result.

In [ ]:
summer = (trips
          .filter(F.col("pickup_month").between(6, 8))
          .join(F.broadcast(pu_zones), "PULocationID")
          .withColumn("total_per_minute", F.expr("try_divide(total_amount, duration_min)")))

def three_actions(df):
    df.groupBy("pu_borough").count().collect()
    df.groupBy("pickup_hour").agg(F.avg("total_per_minute")).collect()
    df.agg(F.avg("tip_pct"), F.max("trip_distance")).collect()

timed("3 actions, NO cache", lambda: three_actions(summer))

try:
    summer_cached = summer.cache()
    timed("Fill cache (first action)", summer_cached.count)
    timed("3 actions, CACHED", lambda: three_actions(summer_cached))
    summer_cached.unpersist()
except Exception as e:
    print(f"\n.cache() is not available here: {type(e).__name__}: {str(e)[:200]}")
    print("Falling back to materializing the subset as a Delta table.\n")
    MAT_TABLE = f"{CATALOG}.{SCHEMA}.taxi_summer_materialized"
    timed("Materialize subset to Delta", lambda: summer.write.mode("overwrite").saveAsTable(MAT_TABLE))
    timed("3 actions on materialized table", lambda: three_actions(spark.table(MAT_TABLE)))

## 10. Write results

Final aggregates go to **Parquet** files in a Unity Catalog volume. They are small, so `coalesce(1)` writes one file each instead of many tiny ones.
The large cleaned dataset is already saved as the partitioned Delta table `taxi_trips_2019` (section 4).

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")

results = {
    "borough_stats":         borough_stats,
    "hourly_patterns":       hourly_patterns,
    "top_pickup_zones":      top_zones,
    "airport_monthly":       airport_monthly,
    "top_route_per_borough": top_route_per_borough,
}
for name, df in results.items():
    timed(f"write {name}", lambda: df.coalesce(1).write.mode("overwrite").parquet(f"{OUTPUT_DIR}/{name}"))

display(dbutils.fs.ls(OUTPUT_DIR))

In [ ]:
# Read one result back to prove the write worked
display(spark.read.parquet(f"{OUTPUT_DIR}/borough_stats"))

## 11. Machine learning with MLlib (optional)

**Task: regression.** Predict a trip's `fare_amount` from what is known about the trip: distance, duration, hour of day, weekend flag and whether it started at an airport.
We use a 2% random sample of the clean Delta table (about 1.6M trips) so training takes seconds, split 80/20 into train and test sets.
`VectorAssembler` packs the feature columns into one vector and `LinearRegression` learns a weight for each; both run as one `Pipeline`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

FEATURES = ["trip_distance", "duration_min", "hour", "weekend", "airport_pickup"]
AIRPORT_ZONES = [1, 132, 138]   # Newark, JFK, LaGuardia

ml_data = (trips
    .sample(fraction=0.02, seed=42)
    .withColumn("hour",           F.col("pickup_hour").cast("double"))
    .withColumn("weekend",        F.col("is_weekend").cast("double"))
    .withColumn("airport_pickup", F.col("PULocationID").isin(AIRPORT_ZONES).cast("double"))
    .select(*FEATURES, "fare_amount")
    .dropna())

train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"train: {train.count():,} rows   test: {test.count():,} rows")

In [ ]:
pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURES, outputCol="features"),
    LinearRegression(featuresCol="features", labelCol="fare_amount", maxIter=20),
])
model = timed("Train LinearRegression", lambda: pipeline.fit(train))

predictions = model.transform(test)
evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")
rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae  = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2   = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
print(f"Test RMSE: ${rmse:.2f}   MAE: ${mae:.2f}   R^2: {r2:.3f}")

lr_model = model.stages[-1]
print(f"\nIntercept: ${lr_model.intercept:.2f}")
for name, coef in zip(FEATURES, lr_model.coefficients):
    print(f"  {name:<15} {coef:+.3f}")

In [ ]:
display(predictions
        .select("trip_distance", "duration_min", "airport_pickup", "fare_amount",
                F.round("prediction", 2).alias("predicted_fare"))
        .orderBy(F.rand(seed=7))   # random rows, not just the shortest trips
        .limit(10))

## 12. Key findings

1. **Manhattan accounts for 91% of yellow-taxi trips** (74.1M of 81.2M). Its trips are short (2.4 mi, 13 min) and slow (10.5 mph); Queens trips, mostly from the airports, average 11.7 mi and a $36 fare.
2. **Midtown traffic shows up in the speeds:** weekday taxis average 9.0-9.6 mph between 8 am and 6 pm vs 20.3 mph at 4 am, and fare per mile rises from about $5.10 overnight to about $7.30 at midday.
3. **The busiest route is Upper East Side South -> Upper East Side North** (524,760 trips). The busiest route out of Queens is LaGuardia -> Times Square (123,957).
4. **JFK is the most expensive common pickup** (about $55-60, 16 mi, 36-43 min) and the 8th-busiest zone; LaGuardia trips are about 9 mi and $41-44. Newark pickups are rare because Newark is outside the yellow-cab pickup area.
5. **Tipping depends on borough:** card payers tip 25.3% in Manhattan, 21.1% in Queens, 16.9% in Brooklyn and 3.2% in the Bronx. About 72% of trips are paid by card.
6. **Data quality:** 3.8% of raw records were invalid (pickups outside 2019, non-positive distances, fares below the $2.50 minimum, impossible durations or speeds).

**Performance:** partitioned Delta answered the same query 33x faster than raw CSV (29.9 s -> 0.9 s), and a broadcast join beat a forced sort-merge join 3.9x (3.43 s -> 0.89 s).

## 13. Clean up

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TRIPS_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.taxi_summer_materialized")
# spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
# print("Dropped")